# Building an xBA model

The first thing I will do is create a dataset where each row represents a batted ball, and the target is whether or not the outcome was a hit. Batted balls include: single, double, triple, home run, field out, force out, double play, error, and fielders choice. The model will learn the relationship between contact quality and outcomes.

To start, we will use launch speed (exit velocity), launch angle, spray direction, batter/pitcher handedness, and pitch information. We will keep the sample small while developing the pipeline, then expand the date range for evaluation.

In [8]:
# Step 1: Download a larger Statcast sample for development, then cache it locally.
# The cache prevents repeated API requests every time the notebook is re-run.
from pathlib import Path

import pandas as pd
from pybaseball import statcast

START_DATE = "2024-04-01"
END_DATE = "2024-05-31"
CACHE_DIR = Path("data")
CACHE_PATH = CACHE_DIR / f"statcast_{START_DATE}_{END_DATE}.parquet"

if CACHE_PATH.exists():
    # Reuse the existing extract when this exact date range has already been downloaded.
    raw = pd.read_parquet(CACHE_PATH)
    print(f"Loaded {len(raw):,} pitches from {CACHE_PATH}")
else:
    # Download the date range once and save the raw response for reproducibility.
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    raw = statcast(START_DATE, END_DATE)
    raw.to_parquet(CACHE_PATH, index=False)
    print(f"Downloaded and cached {len(raw):,} pitches to {CACHE_PATH}")

# Step 2: Keep only events where the ball was put into play and define the hit target.
batted_ball_events = {
    "single", "double", "triple", "home_run", "field_out",
    "force_out", "grounded_into_double_play", "field_error",
    "fielders_choice", "fielders_choice_out", "double_play"
}

contact = raw.loc[
    raw["events"].isin(batted_ball_events),
    [
        "game_date", "batter", "events", "launch_speed", "launch_angle",
        "hc_x", "hc_y", "stand", "p_throws", "pitch_type",
    ],
].copy()

# The target is 1 for any hit and 0 for every other retained batted-ball outcome.
contact["is_hit"] = contact["events"].isin(
    {"single", "double", "triple", "home_run"}
).astype(int)

contact.head()

Loaded 235,508 pitches from data/statcast_2024-04-01_2024-05-31.parquet


,game_date,batter,events,launch_speed,launch_angle,hc_x,hc_y,stand,p_throws,pitch_type,is_hit
0,2024-05-31,647351,field_out,73.9,14,116.13,131.52,L,R,FF,0
1,2024-05-31,649557,field_out,104.6,6,102.71,143.47,R,R,SI,0
2,2024-05-31,691016,field_out,87.7,-31,153.3,147.02,L,R,FF,0
3,2024-05-31,680869,double,92.9,25,38.25,113.72,R,R,CH,1
6,2024-05-31,594807,field_out,90.7,5,135.42,142.46,R,L,FF,0


In [9]:
# Step 3: Remove rows missing features required by the first model.
# We do this after defining the target so missingness cannot silently change the label.
model_data = contact.dropna(
    subset=[
        "launch_speed", "launch_angle", "hc_x", "hc_y",
        "stand", "p_throws", "pitch_type", "is_hit",
    ]
).copy()

print(f"Contact events: {len(contact):,}")
print(f"Complete modeling rows: {len(model_data):,}")
print(f"Observed batting average: {model_data['is_hit'].mean():.3f}")
print("Missing values by field:")
print(contact.isna().sum().sort_values(ascending=False).head(8))

Contact events: 40,653
Complete modeling rows: 40,520
Observed batting average: 0.318
Missing values by field:
launch_speed    119
launch_angle    107
pitch_type       23
hc_x             14
hc_y             14
game_date         0
batter            0
events            0
dtype: int64


In [10]:
# Step 4: Split by date so the model is tested on a later day than it saw during training.
# Training on all earlier dates gives the model enough examples while preserving temporal order.
model_data["game_date"] = pd.to_datetime(model_data["game_date"])
cutoff_date = pd.Timestamp(END_DATE)
train_data = model_data[model_data["game_date"] < cutoff_date].copy()
test_data = model_data[model_data["game_date"] == cutoff_date].copy()

numeric_features = ["launch_speed", "launch_angle", "hc_x", "hc_y"]
categorical_features = ["stand", "p_throws", "pitch_type"]
feature_columns = numeric_features + categorical_features

X_train = train_data[feature_columns]
y_train = train_data["is_hit"]
X_test = test_data[feature_columns]
y_test = test_data["is_hit"]

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training dates: {train_data['game_date'].min().date()} to {train_data['game_date'].max().date()}")
print(f"Test date: {test_data['game_date'].iloc[0].date()}")

Training rows: 39,727
Test rows: 793
Training dates: 2024-04-01 to 2024-05-30
Test date: 2024-05-31


In [11]:
# Step 5: Build an interpretable logistic-regression baseline.
# One-hot encoding lets the model use categorical features without treating labels as numbers.
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1_000, random_state=42)),
    ]
)

logistic_model.fit(X_train, y_train)

# Predict probabilities rather than only hit/out labels because xBA is an average of probabilities.
test_data["predicted_hit_probability"] = logistic_model.predict_proba(X_test)[:, 1]
test_data[["game_date", "batter", "is_hit", "predicted_hit_probability"]].head()

,game_date,batter,is_hit,predicted_hit_probability
0,2024-05-31,647351,0,0.192026
1,2024-05-31,649557,0,0.229268
2,2024-05-31,691016,0,0.389934
3,2024-05-31,680869,1,0.259270
6,2024-05-31,594807,0,0.207516


In [12]:
# Step 6: Evaluate probability quality on the held-out date.
# Log loss and Brier score reward useful probabilities; accuracy alone would hide calibration errors.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
baseline_probabilities = baseline.predict_proba(X_test)[:, 1]
model_probabilities = test_data["predicted_hit_probability"]

metrics = pd.DataFrame(
    {
        "log_loss": [
            log_loss(y_test, baseline_probabilities),
            log_loss(y_test, model_probabilities),
        ],
        "brier_score": [
            brier_score_loss(y_test, baseline_probabilities),
            brier_score_loss(y_test, model_probabilities),
        ],
        "accuracy_at_0.5": [
            accuracy_score(y_test, baseline_probabilities >= 0.5),
            accuracy_score(y_test, model_probabilities >= 0.5),
        ],
    },
    index=["overall_hit_rate_baseline", "logistic_regression"],
)

metrics

,log_loss,brier_score,accuracy_at_0.5
overall_hit_rate_baseline,0.629995,0.219091,0.675914
logistic_regression,0.531818,0.179678,0.720050


In [13]:
# Step 7: Aggregate contact-level probabilities into a batter-level expected batting average.
# A batter's xBA is the mean predicted hit probability across that batter's batted balls.
batter_summary = (
    test_data.groupby("batter")
    .agg(
        batted_balls=("is_hit", "size"),
        observed_batting_average=("is_hit", "mean"),
        expected_batting_average=("predicted_hit_probability", "mean"),
    )
    .sort_values("batted_balls", ascending=False)
)

batter_summary.head(10)

,batted_balls,observed_batting_average,expected_batting_average
batter,,,
457705,6,0.333333,0.354011
701538,5,0.400000,0.456313
677951,5,0.600000,0.442407
641355,5,0.000000,0.237288
624413,5,0.200000,0.279224
668901,5,0.600000,0.286736
668939,5,0.200000,0.484581
669394,5,0.600000,0.278870
595777,5,0.600000,0.310113
